In [1]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile


In [2]:
#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']


Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



In [3]:
# %pip install tensorflow
# %pip install keras
# %pip install scikit-learn
# %pip install fastai
# %pip install torch

In [4]:
#Import required libraries for Deep Learning
#Keras DNN Classifier
from keras.models import Sequential
from keras.layers import BatchNormalization, Dense, Dropout
from keras.regularizers import l2
from keras.utils import to_categorical, normalize
from keras import backend as K

#FastAI DL Classifier
import torch
from fastai.tabular.all import *

#Metrics for evaluation
from sklearn.metrics import balanced_accuracy_score, accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [5]:
#Display the info of the xuetangx_df DataFrame
xuetangx_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225642 entries, 0 to 225641
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   enroll_id                      225642 non-null  int64  
 1   action_count                   225642 non-null  float64
 2   seek_video_count               225642 non-null  float64
 3   play_video_count               225642 non-null  float64
 4   pause_video_count              225642 non-null  float64
 5   stop_video_count               225642 non-null  float64
 6   load_video_count               225642 non-null  float64
 7   problem_get_count              225642 non-null  float64
 8   problem_check_count            225642 non-null  float64
 9   problem_save_count             225642 non-null  float64
 10  reset_problem_count            225642 non-null  float64
 11  problem_check_correct_count    225642 non-null  float64
 12  problem_check_incorrect_count 

In [6]:
#Display the info of the kdd_df DataFrame
kdd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200904 entries, 0 to 200903
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   enrollment_id             200904 non-null  int64  
 1   action_count              200904 non-null  float64
 2   server_navigate_count     200904 non-null  float64
 3   server_access_count       200904 non-null  float64
 4   server_problem_count      200904 non-null  float64
 5   server_page_close_count   200904 non-null  float64
 6   server_video_count        200904 non-null  float64
 7   server_discussion_count   200904 non-null  float64
 8   server_wiki_count         200904 non-null  float64
 9   browser_navigate_count    200904 non-null  float64
 10  browser_access_count      200904 non-null  float64
 11  browser_problem_count     200904 non-null  float64
 12  browser_page_close_count  200904 non-null  float64
 13  browser_video_count       200904 non-null  f

In [7]:
#Display the info of the kdd_expanded_df DataFrame
kdd_expanded_df.columns

Index(['Unnamed: 0', 'enrollment_id', 'truth', 'avg_chapter_delays',
       'server_discussion_percent', 'act_cnt_weekDay_01',
       'browser_html_percent', 'parallel_enrollments', 'browser_dictation',
       'act_cnt_day_00',
       ...
       'server_course_percent', 'browser_course_info_percent',
       'browser_course', 'browser_vertical_percent', 'sessions_in_week_1',
       'sessions_in_week_0', 'sessions_in_week_3', 'sessions_in_week_2',
       'sessions_in_week_4', 'browser_about'],
      dtype='object', length=142)

In [8]:
#Isolate the X and y features for the xuetangx dataset
xuetangx_X = xuetangx_df.drop(columns=['truth'])
xuetangx_X = xuetangx_X.drop(columns=['enroll_id'])
xuetangx_y = xuetangx_df['truth']
#Isolate the X and y features for the kdd dataset
kdd_X = kdd_df.drop(columns=['truth'])
kdd_X = kdd_X.drop(columns=['enrollment_id'])
kdd_y = kdd_df['truth']
#Isolate the X and y features for the kdd_expanded 
kdd_expanded_X = kdd_expanded_df.drop(columns=['truth'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['enrollment_id'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['Unnamed: 0'])
kdd_expanded_y = kdd_expanded_df['truth']

In [9]:
#Split the xuetangx dataset into training and testing sets
xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test = train_test_split(xuetangx_X, xuetangx_y, test_size=0.2, random_state=100)

#Split the kdd dataset into training and testing sets
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_y, test_size=0.2, random_state=100)

#Split the kdd_expanded dataset into training and testing sets
kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test = train_test_split(kdd_expanded_X, kdd_expanded_y, test_size=0.2, random_state=100)

### Keras-TensorFlow

#### Xuetangx

In [10]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [11]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
xuetang_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(xuetang_results_df)

1411/1411 ━━━━━━━━━━━━━━━━━━━━ 1s 555us/step
              Metric      Value
0           Accuracy  84.101132
1  Balanced Accuracy  72.858679
2             Recall  84.101132
3          Precision  83.236705
4                AUC  72.858679
5           F1 Score  90.034030


#### KDD (Experiment 1)

In [13]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [14]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_results_df)

1256/1256 ━━━━━━━━━━━━━━━━━━━━ 1s 537us/step
              Metric      Value
0           Accuracy  86.043155
1  Balanced Accuracy  73.333619
2             Recall  86.043155
3          Precision  85.123587
4                AUC  73.333619
5           F1 Score  91.522041


#### KDD (Experiment 2)

In [16]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [17]:
#Create a DNN model using Keras
dnn_keras = Sequential(layers=[
                            Dense(128, kernel_regularizer=l2(0.001), activation='relu',input_shape=(len(X_train.columns),)),
                            BatchNormalization(),
                            Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
                            BatchNormalization(),
                            Dense(y_train.nunique(), activation='softmax')
])
dnn_keras.compile(
    optimizer='adam', 
    loss='binary_crossentropy')

dnn_keras.fit(X_train, pd.get_dummies(y_train), epochs=100, verbose=0, batch_size=512)
#loss, acc = dnn_keras.evaluate(X_test, pd.get_dummies(y_test), verbose=0)

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Create predicted values for the test set
y_pred = np.argmax(dnn_keras.predict(X_test), axis=1)

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_expanded_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_expanded_results_df)

754/754 ━━━━━━━━━━━━━━━━━━━━ 0s 539us/step
              Metric      Value
0           Accuracy  85.860052
1  Balanced Accuracy  74.626745
2             Recall  85.860052
3          Precision  85.042046
4                AUC  74.626745
5           F1 Score  91.317968


### FastAI

#### Xuetangx

In [19]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [20]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(xuetangx_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

epoch,train_loss,valid_loss,accuracy,time
0,0.135350,0.241567,0.235029,00:10
1,0.125083,0.134298,0.235029,00:10
2,0.125361,0.143529,0.235029,00:09
3,0.124398,0.306945,0.235029,00:09
4,0.120781,0.128209,0.235029,00:09


In [ ]:
#Convert X_test to a TabularDataLoader for batch predictions
dl = dnn_fastai.dls.test_dl(X_test)

#Get predictions on the entire test set
preds, _, _ = dnn_fastai.get_preds(dl=dl, with_decoded=True)

#Convert predictions to binary (0 or 1) using a threshold of 0.5
y_pred = (preds >= 0.5).int().tolist()

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
xuetangx_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(xuetangx_fastai_results_df)

              Metric      Value
0           Accuracy  83.779831
1  Balanced Accuracy  71.010059
2             Recall  83.779831
3          Precision  82.990418
4                AUC  71.010059
5           F1 Score  89.956643


#### KDD (Experiment 1)

In [ ]:
#Modfy the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [27]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(kdd_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

epoch,train_loss,valid_loss,accuracy,time
0,0.130441,0.125098,0.241849,00:08
1,0.120106,0.121987,0.241849,00:08
2,0.118258,0.120294,0.241849,00:08
3,0.121510,0.121303,0.241849,00:08
4,0.120511,0.118504,0.241849,00:08


In [ ]:
#Convert X_test to a TabularDataLoader for batch predictions
dl = dnn_fastai.dls.test_dl(X_test)

#Get predictions on the entire test set
preds, _, _ = dnn_fastai.get_preds(dl=dl, with_decoded=True)

#Convert predictions to binary (0 or 1) using a threshold of 0.5
y_pred = (preds >= 0.5).int().tolist()

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_fastai_results_df)

              Metric      Value
0           Accuracy  85.998357
1  Balanced Accuracy  73.460281
2             Recall  85.998357
3          Precision  85.078235
4                AUC  73.460281
5           F1 Score  91.483500


#### KDD (Experiment 2)

In [ ]:
#Modfy the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [30]:
#Create a DNN model using FastAI
splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
feature_set = X_train.columns
y_names = ['truth']
#print(df[:5])
tp = TabularPandas(kdd_expanded_df, procs=[],
                    cat_names= [],
                    cont_names = list(feature_set),
                    y_names= y_names,
                    splits=splits)

dls = tp.dataloaders(bs=64)
#dls.show_batch()
#return
dnn_fastai = tabular_learner(dls, metrics=accuracy)
dnn_fastai.fit_one_cycle(5)

epoch,train_loss,valid_loss,accuracy,time
0,0.113319,3.739652,0.238411,00:05
1,0.107269,0.111777,0.238411,00:05
2,0.105148,0.154051,0.238411,00:05
3,0.100301,0.106935,0.238411,00:05
4,0.101607,0.107825,0.238411,00:05


In [ ]:
#Convert X_test to a TabularDataLoader for batch predictions
dl = dnn_fastai.dls.test_dl(X_test)

#Get predictions on the entire test set
preds, _, _ = dnn_fastai.get_preds(dl=dl, with_decoded=True)

#Convert predictions to binary (0 or 1) using a threshold of 0.5
y_pred = (preds >= 0.5).int().tolist()

#Generate metrics and convert to percentage
acc = accuracy_score(y_test, y_pred) * 100
bal_acc = balanced_accuracy_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred, average='weighted') * 100
prec = precision_score(y_test, y_pred, average='weighted') * 100
auc = roc_auc_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

#Store the results in a dictionary
results = {
    'Accuracy': acc,
    'Balanced Accuracy': bal_acc,
    'Recall': rec,
    'Precision': prec,
    'AUC': auc,
    'F1 Score': f1
}
#Convert the results dictionary to a DataFrame
kdd_expanded_fastai_results_df = pd.DataFrame(list(results.items()), columns=['Metric', 'Value'])
#Display the results DataFrame
print(kdd_expanded_fastai_results_df)

              Metric      Value
0           Accuracy  87.880045
1  Balanced Accuracy  76.735167
2             Recall  87.880045
3          Precision  87.256615
4                AUC  76.735167
5           F1 Score  92.607398


#### SVM

In [11]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score

In [12]:
def shorttraintest(vartrain, vartest, y_train, y_test, model):

    #Fit the model
    model.fit(vartrain, y_train)

    #Predict with the model
    model_pred = model.predict(vartest)
    model_prob = model.predict_proba(vartest)


    print('Confusion Matrix:')
    print(confusion_matrix(y_test, model_pred))
    print("")

    #Assess with the model
    score = model.score(vartest, y_test)
    score_format = 'Accuracy Score: {0:.4f}'.format(score)
    print(score_format)

    recall = recall_score(y_test, model_pred)
    recall_format = 'Recall Score: {0:.4f}'.format(recall)
    print(recall_format)
    
    precision = precision_score(y_test, model_pred)
    precision_format = 'Precision Score: {0:.4f}'.format(precision)
    print(precision_format)
    
    # calculate roc curve
    y_pred_prob = model.predict_proba(vartest)[:,1]
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    roc_auc_format = 'ROC AUC Score: {0:.4f}'.format(roc_auc)
    print(roc_auc_format)
    print('')

### Xuetangx SVM

In [13]:
#Modfy the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [14]:
svc = SVC(gamma='auto')
scores = cross_val_score(svc, X_train, y_train, cv=5, n_jobs=-1)

In [15]:
scores

array([0.83142675, 0.8342797 , 0.83347644, 0.83291784, 0.83527228])

In [35]:
#Create an SVM model
svm_model = SVC(gamma='auto')
shorttraintest(X_train, X_test, y_train, y_test, svm_model)

: 

: 

#### KDD (Experiment 1) SVM

In [ ]:
#Modfy the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [ ]:
#Create an SVM model
svm_model = SVC(gamma='auto')
shorttraintest(X_train, X_test, y_train, y_test, svm_model)

### KDD (Experiment 2) SVM

In [ ]:
#Modfy the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [ ]:
#Create an SVM model
svm_model = SVC(gamma='auto')
shorttraintest(X_train, X_test, y_train, y_test, svm_model)